# OPENBRIDGE — LABORATORIO DE INVESTIGACIÓN EMPÍRICA CON GITHUB REST API (DATOS REALES)

**Módulo:** Data Engineering / ETL Exploratorio  
**Fase:** 2 — Separación de Experimentos Simulados e Investigación Empírica  
**Entorno de Ejecución:** Python 3.11.9 (`.venv` del proyecto OpenBridge)  
**Estado Operativo:** **BLOQUEO DE RED ACTIVO** (`ENABLE_GITHUB_EXTRACTION = False`)  
**Persistencia Externa:** **DESACTIVADA** (Sin conexiones ni escrituras en PostgreSQL)  

---

## 1. Introducción y Objetivo del Laboratorio

### 1.1 Contexto de OpenBridge
OpenBridge es un proyecto de ingeniería de datos y software inteligente diseñado para **investigar, catalogar, normalizar y relacionar metadatos de software open source** procedentes de diversos ecosistemas tecnológicos. El catálogo unificado resultante servirá como fuente canónica estructurada para alimentar un sistema de recomendación asistido por inteligencia artificial (RAG — *Retrieval-Augmented Generation*).

### 1.2 Propósito Específico de esta Segunda Notebook
A diferencia de la notebook de diseño y exploración inicial (`01_exploracion_fuentes.ipynb`), que incorporaba bancos de pruebas simuladas (*fixtures*) para evaluar resiliencia algorítmica sin requerir conectividad:
1. **Aislamiento Empírico:** Esta notebook está destinada **exclusivamente a interactuar con la API real de GitHub** y a analizar datos empíricos observados.
2. **Sin Sustitución de Datos:** Los resultados reales **nunca se sustituyen por fixtures ni datos sintéticos** si ocurre una falla de red. Si una petición falla, el registro queda asentado como fallido en la bitácora de auditoría y no ingresa al catálogo analítico.
3. **Reproducibilidad y Rigor Metodológico:** Cada paso (extracción, transformación, auditoría operativa, validación contractual, exploración estadística y análisis de relaciones) está documentado en español técnico con controles explícitos de seguridad y defensas ante respuestas vacías.

## 2. Fuente y Alcance de la Investigación

### 2.1 Fuente Tecnológica Exclusiva
La investigación empírica se restringe estrictamente a **GitHub REST API (v3 / versión 2022-11-28)**.  
No se consultan registros de paquetes (PyPI, npm, Maven, crates.io), ni bases de vulnerabilidades (OSV.dev), ni catálogos de licencias (SPDX) en esta etapa.

### 2.2 Muestra Aprobada de Repositorios (5 Proyectos)
Se ha seleccionado una muestra diversa de 5 repositorios públicos con alta relevancia técnica y tipologías arquitectónicas heterogéneas:

| N° | Repositorio (`owner/name`) | Propietario | Tipo de Propietario | Lenguaje Principal | Dominio Arquitectónico | Justificación de Selección |
|---|---|---|---|---|---|---|
| 1 | `apache/airflow` | `apache` | Organization | Python | Orquestación / Data Eng | Proyecto insignia de fundación de software, gobernanza abierta, múltiples dependencias. |
| 2 | `facebook/react` | `facebook` | Organization | JavaScript | Frontend / Web UI | Proyecto masivo corporativo, ecosistema npm, alto volumen de issues y forks. |
| 3 | `torvalds/linux` | `torvalds` | User | C | Sistemas / Kernel OS | Repositorio histórico masivo, propiedad de cuenta de usuario personal, sin topics nativos. |
| 4 | `prometheus/prometheus` | `prometheus` | Organization | Go | Observabilidad / Métricas | Proyecto graduado de Cloud Native Computing Foundation (CNCF), monitoreo en la nube. |
| 5 | `BurntSushi/ripgrep` | `BurntSushi` | User | Rust | CLI / Utilidades de Sistema | Proyecto de alto rendimiento en Rust, mantenido por desarrollador individual respetado. |

### 2.3 Delimitación Estricta de Endpoints (Sin Búsquedas ni Paginación)
Únicamente se autorizan los 5 endpoints canónicos correspondientes a la entidad `Repository`:
- `https://api.github.com/repos/apache/airflow`
- `https://api.github.com/repos/facebook/react`
- `https://api.github.com/repos/torvalds/linux`
- `https://api.github.com/repos/prometheus/prometheus`
- `https://api.github.com/repos/BurntSushi/ripgrep`

> **Límite operativo:** Máximo 5 peticiones HTTP GET por ciclo de ejecución autorizado. No se permite paginación, navegación de ramas ni endpoints de colaboradores/commits.

## 3. Contrato de Datos Esperado y Especificación Técnica

Para garantizar que el catálogo de OpenBridge mantenga integridad referencial y estructural, se establece el siguiente contrato preliminar de campos mínimos requeridos:

| Campo | Tipo JSON | Tipo PostgreSQL Propuesto | Obligatorio | Nulable | Origen en Payload | Regla de Validación Contractual |
|---|---|---|---|---|---|---|
| `id` | `int` | `BIGINT PRIMARY KEY` | Sí | No | Raíz `id` | Entero positivo inmutable de 64 bits. **Rechazo explícito de booleanos**. |
| `name` | `str` | `VARCHAR(100)` | Sí | No | Raíz `name` | Cadena no vacía; nombre del repo sin namespace. |
| `full_name` | `str` | `VARCHAR(255) UNIQUE` | Sí | No | Raíz `full_name` | Formato estricto `owner/repo`. Clave natural candidata. |
| `html_url` | `str` | `TEXT` | Sí | No | Raíz `html_url` | URL HTTPS bien formada hacia la interfaz web de GitHub. |
| `description` | `str` / `null` | `TEXT` | No | Sí | Raíz `description` | Cadena o nulo; resumen declarativo del software. |
| `language` | `str` / `null` | `VARCHAR(50)` | No | Sí | Raíz `language` | Cadena o nulo; lenguaje principal inferido por Linguist. |
| `stargazers_count` | `int` | `INTEGER` | Sí | No | Raíz `stargazers_count` | Entero no negativo $\ge 0$. Métrica de popularidad. |
| `forks_count` | `int` | `INTEGER` | Sí | No | Raíz `forks_count` | Entero no negativo $\ge 0$. Métrica de derivación. |
| `open_issues_count`| `int` | `INTEGER` | Sí | No | Raíz `open_issues_count` | Entero no negativo $\ge 0$. Métrica de actividad/deuda. |
| `subscribers_count`| `int` | `INTEGER` | No | Sí | Raíz `subscribers_count` | Entero no negativo $\ge 0$ (observadores activos). |
| `created_at` | `str` (ISO 8601) | `TIMESTAMPTZ` | Sí | No | Raíz `created_at` | Marca temporal UTC en formato ISO 8601 (`YYYY-MM-DDTHH:MM:SSZ`). |
| `updated_at` | `str` (ISO 8601) | `TIMESTAMPTZ` | Sí | No | Raíz `updated_at` | Marca temporal UTC; última actualización de metadatos. |
| `pushed_at` | `str` (ISO 8601) | `TIMESTAMPTZ` | Sí | No | Raíz `pushed_at` | Marca temporal UTC; último commit a cualquier rama. |
| `topics` | `list` [`str`] | `TEXT[]` | No | No (vacío si no hay) | Raíz `topics` | Arreglo JSON de strings en minúsculas. |
| `license.spdx_id` | `str` / `null` | `VARCHAR(50)` | No | Sí | Objeto `license` | Identificador canónico de licencia según especificación SPDX. |
| `owner.id` | `int` | `BIGINT` | Sí | No | Objeto `owner` | Identificador unívoco del propietario en GitHub. |
| `owner.login` | `str` | `VARCHAR(100)` | Sí | No | Objeto `owner` | Nombre de usuario u organización propietaria. |

### 3.1 Identificador Primario: Inmutabilidad de `id` vs Mutabilidad de `full_name`
- **`id` (GitHub ID):** Es un identificador entero global generado secuencialmente por GitHub. Permanece invariable ante transferencias de propiedad o renombrados del proyecto. Constituye la mejor opción para la clave única primaria.
- **`full_name` (`owner/repo`):** Es una clave natural intuitiva para usuarios y APIs, pero es **mutable** (por ejemplo, si un proyecto cambia de nombre o se transfiere de un desarrollador personal a una organización). En OpenBridge debe manejarse con restricción `UNIQUE`, pero vinculada a un registro histórico de cambios.

## 4. Configuración Segura y Parámetros Operativos

En esta sección se inicializan las bibliotecas requeridas, se definen las políticas de seguridad de red y se configura el acceso defensivo a la API.

### 4.1 Principios de Seguridad Aplicados
1. **Guarda Central de Red (`ENABLE_GITHUB_EXTRACTION = False`):** Ninguna petición HTTP se emitirá hacia GitHub hasta que el usuario autorice expresamente el cambio de esta variable a `True`.
2. **Lista Blanca Estricta (`APPROVED_GITHUB_TARGETS`):** Toda URL debe coincidir exactamente con una clave aprobada antes de disparar `requests.get()`.
3. **Manejo Seguro de Credenciales:** La celda comprueba si existe una variable de entorno `GITHUB_TOKEN`. Si está presente, se añade como encabezado de autorización `Bearer` para incrementar la cuota horaria de 60 a 5.000 peticiones por hora. **Nunca se almacena, imprime ni expone el token en pantalla ni en bitácoras.**
4. **Protección contra Bucles y Colgamientos:** `allow_redirects = False` y `timeout = 10` segundos.

In [ ]:
# Celda de Configuración Segura del Laboratorio
import os
import sys
import json
import time
import re
from datetime import datetime, timezone
from typing import List, Dict, Any, Tuple, Optional
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

# 1. GUARDA CENTRAL DE SEGURIDAD (Deshabilitada por defecto)
# ADVERTENCIA: Mantener en False hasta recibir autorización formal de ejecución.
ENABLE_GITHUB_EXTRACTION: bool = False

# 2. CONSTANTES TÉCNICAS Y OPERATIVAS
HTTP_TIMEOUT_SECONDS: int = 10
USER_AGENT: str = "OpenBridge-Empirical-Research/2.0 (contact: data-eng@openbridge.local)"

# 3. LISTA BLANCA ESTRICTA DE LOS 5 ENDPOINTS APROBADOS
APPROVED_GITHUB_TARGETS: Dict[str, str] = {
    "apache/airflow": "https://api.github.com/repos/apache/airflow",
    "facebook/react": "https://api.github.com/repos/facebook/react",
    "torvalds/linux": "https://api.github.com/repos/torvalds/linux",
    "prometheus/prometheus": "https://api.github.com/repos/prometheus/prometheus",
    "BurntSushi/ripgrep": "https://api.github.com/repos/BurntSushi/ripgrep",
}

# 4. GESTIÓN SEGURA DE AUTENTICACIÓN (Lectura opcional de variable de entorno)
GITHUB_TOKEN_ENV_VAR = "GITHUB_TOKEN"
_token_candidate: Optional[str] = os.getenv(GITHUB_TOKEN_ENV_VAR)
IS_AUTHENTICATED: bool = bool(_token_candidate and _token_candidate.strip())

print("================================================================================")
print(f"[*] Entorno Python activo: {sys.version.split()[0]}")
print(f"[*] Guarda de Extracción de Red: {'[HABILITADA]' if ENABLE_GITHUB_EXTRACTION else '[BLOQUEADA POR SEGURIDAD]'}")
print(f"[*] Estado de Autenticación Token: {'Detectado en Entorno (Cuota: ~5000 req/h)' if IS_AUTHENTICATED else 'No detectado (Anónimo - Cuota: ~60 req/h)'}")
print(f"[*] Endpoints autorizados en lista blanca: {len(APPROVED_GITHUB_TARGETS)}")
print("================================================================================")

## 5. Extracción Resiliente y Controlada (EXTRACT)

### 5.1 Protocolo de Solicitudes HTTP
- **Método:** `GET` exclusivo.
- **Volumen máximo:** 5 solicitudes por ejecución completa.
- **Redirecciones:** `allow_redirects=False`. Cualquier código `301`, `302`, `307` o `308` se bloquea y se asienta como anomalía para evitar redirecciones imprevistas o secuestros de dominio.
- **Manejo individual de excepciones:** Si una consulta falla por timeout, DNS o código de estado HTTP (p. ej. `404`), las demás peticiones continúan normalmente.
- **Diferenciación de Errores 403:** Se inspeccionan las cabeceras `x-ratelimit-remaining` y el cuerpo del mensaje para distinguir un bloqueo por agotamiento de cuota (`HTTPError_403_RateLimitExceeded`) de una denegación por permisos o firewall (`HTTPError_403_Forbidden`).
- **Verificación Estructural de JSON:** Se comprueba que la respuesta sea un diccionario de Python válido y que contenga los campos nucleares `id` y `full_name`.
- **Linaje y Metadatos de Extracción:** A cada registro exitoso se le inyectan metadatos operativos (`_metadata_extracted_at_utc`, `_metadata_endpoint_url`, `_metadata_http_status`, `_metadata_source`, `_metadata_rate_limit_remaining`, `_metadata_is_simulated = False`).

In [ ]:
# Celda de Extracción Empírica Resiliente
def classify_http_403_error(response) -> str:
    """Distingue agotamiento de cuota horaria de otros errores HTTP 403."""
    remaining = response.headers.get("x-ratelimit-remaining") if hasattr(response, "headers") else None
    body = response.text.lower() if hasattr(response, "text") else ""
    if remaining == "0" or "rate limit" in body or "rate_limit" in body:
        return "HTTPError_403_RateLimitExceeded"
    return "HTTPError_403_Forbidden"

def extract_github_repositories(
    targets: Dict[str, str]
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    """
    Ejecuta la extracción empírica controlada hacia GitHub REST API.
    Retorna: (repositorios_exitosos, log_de_auditoria)
    """
    if not ENABLE_GITHUB_EXTRACTION:
        print("[BLOQUEO ACTIVO] ENABLE_GITHUB_EXTRACTION = False. No se emitirán solicitudes a la red.")
        print("-> Para habilitar la extracción real una vez autorizada, establezca ENABLE_GITHUB_EXTRACTION = True.")
        return [], []

    import requests

    headers = {
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
        "User-Agent": USER_AGENT,
    }
    
    # Inyección de token si está presente (sin almacenar ni imprimir el secreto)
    token = os.getenv("GITHUB_TOKEN")
    if token and token.strip():
        headers["Authorization"] = f"Bearer {token.strip()}"

    successful_records: List[Dict[str, Any]] = []
    audit_log: List[Dict[str, Any]] = []

    for repo_key, target_url in targets.items():
        # Verificación estricta contra lista blanca
        if target_url != APPROVED_GITHUB_TARGETS.get(repo_key):
            audit_log.append({
                "repo_key": repo_key,
                "url": target_url,
                "status_code": None,
                "success": False,
                "error_type": "SecurityAllowlistViolation",
                "error_message": "La URL no coincide con la lista blanca preaprobada.",
                "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                "rate_limit_remaining": None,
                "is_simulated": False,
            })
            continue

        log_entry: Dict[str, Any] = {
            "repo_key": repo_key,
            "url": target_url,
            "status_code": None,
            "success": False,
            "error_type": None,
            "error_message": None,
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            "rate_limit_remaining": None,
            "is_simulated": False,
        }

        try:
            print(f"-> [HTTP GET] Solicitando {repo_key}: {target_url} (allow_redirects=False)...")
            response = requests.get(
                target_url,
                headers=headers,
                timeout=HTTP_TIMEOUT_SECONDS,
                allow_redirects=False
            )
            
            log_entry["status_code"] = response.status_code
            log_entry["rate_limit_remaining"] = response.headers.get("x-ratelimit-remaining")

            if response.is_redirect:
                log_entry["error_type"] = "UnexpectedRedirect"
                log_entry["error_message"] = f"Redirección no autorizada a: {response.headers.get('Location')}"
                print(f"   [AVISO] Redirección rechazada (Código {response.status_code}).")
            elif response.status_code == 403:
                log_entry["error_type"] = classify_http_403_error(response)
                log_entry["error_message"] = response.text[:250]
                print(f"   [ERROR 403] {log_entry['error_type']}.")
            else:
                response.raise_for_status()
                payload = response.json()

                # Validación estructural básica del payload recibido
                if not isinstance(payload, dict):
                    log_entry["error_type"] = "InvalidPayloadStructure"
                    log_entry["error_message"] = f"Se esperaba dict, recibido {type(payload).__name__}"
                    print(f"   [ERROR] Estructura JSON no esperada.")
                elif "id" not in payload or "full_name" not in payload:
                    log_entry["error_type"] = "MissingCoreRepositoryKeys"
                    log_entry["error_message"] = "Faltan claves nucleares (id, full_name) en la respuesta."
                    print(f"   [ERROR] Payload incompleto.")
                else:
                    # Enriquecimiento de linaje y metadatos operativos
                    payload["_metadata_extracted_at_utc"] = log_entry["timestamp_utc"]
                    payload["_metadata_endpoint_url"] = target_url
                    payload["_metadata_http_status"] = response.status_code
                    payload["_metadata_source"] = "GitHub REST API v3 (Live)"
                    payload["_metadata_rate_limit_remaining"] = log_entry["rate_limit_remaining"]
                    payload["_metadata_is_simulated"] = False

                    successful_records.append(payload)
                    log_entry["success"] = True
                    print(f"   [OK] {repo_key} recuperado exitosamente (HTTP {response.status_code}).")

        except requests.exceptions.Timeout as err:
            log_entry["error_type"] = "Timeout"
            log_entry["error_message"] = f"Límite de tiempo excedido ({HTTP_TIMEOUT_SECONDS}s): {err}"
            print(f"   [ERROR: TIMEOUT] {repo_key} no respondió a tiempo.")
        except requests.exceptions.HTTPError as err:
            status = err.response.status_code if err.response is not None else 0
            if status == 404:
                log_entry["error_type"] = "HTTPError_404_NotFound"
            elif status == 403:
                log_entry["error_type"] = classify_http_403_error(err.response)
            else:
                log_entry["error_type"] = f"HTTPError_{status}"
            log_entry["error_message"] = str(err)
            print(f"   [ERROR: HTTP {status}] {repo_key}: {log_entry['error_type']}.")
        except requests.exceptions.ConnectionError as err:
            log_entry["error_type"] = "ConnectionError"
            log_entry["error_message"] = f"Fallo en la conexión de red: {err}"
            print(f"   [ERROR: CONEXIÓN] {repo_key}: No fue posible contactar a GitHub.")
        except json.JSONDecodeError as err:
            log_entry["error_type"] = "JSONDecodeError"
            log_entry["error_message"] = f"Respuesta no parseable como JSON: {err}"
            print(f"   [ERROR: JSON] {repo_key}: Respuesta malformada.")
        except Exception as err:
            log_entry["error_type"] = f"Unexpected_{type(err).__name__}"
            log_entry["error_message"] = str(err)
            print(f"   [ERROR INESPERADO] {repo_key} ({type(err).__name__}): {err}")

        audit_log.append(log_entry)
        time.sleep(0.5)  # Pausa defensiva para proteger la cuota

    return successful_records, audit_log

# Ejecución controlada (Retorna listas vacías mientras ENABLE_GITHUB_EXTRACTION = False)
live_repos_payload, live_audit_log = extract_github_repositories(APPROVED_GITHUB_TARGETS)

## 6. Transformación y Construcción de DataFrames (TRANSFORM)

En esta etapa se procesan exclusivamente las respuestas exitosas obtenidas de GitHub REST API.

### 6.1 Separación Estricta de Colecciones
Para mantener la limpieza analítica y la trazabilidad, se manejan tres DataFrames completamente desacoplados:
1. **`df_repos`:** Contiene los datos aplanados de los repositorios reales recuperados. Si ninguna consulta tuvo éxito o la extracción está bloqueada, permanece como un DataFrame vacío con esquema predefinido.
2. **`df_requests_log`:** Contiene la bitácora operativa de cada solicitud enviada (endpoint, código de estado, tasa de éxito, tipo de error, cuota restante y timestamp UTC).
3. **`df_contract_validation`:** Almacena los resultados de la auditoría de calidad de datos campo por campo.

### 6.2 Aplanado Estructural con `pd.json_normalize()`
La API de GitHub retorna objetos jerárquicos anidados como `owner` (`owner.id`, `owner.login`, `owner.type`), `license` (`license.key`, `license.name`, `license.spdx_id`) y arreglos escalares como `topics`.
- Se emplea `pd.json_normalize()` para desenrollar estas ramas anidadas a columnas tabulares planas de primer nivel usando la convención de punto (`owner.login`, `license.spdx_id`).
- Se preservan intactos los identificadores originales enteros y las cadenas de texto sin mutaciones de contenido.

In [ ]:
# Celda de Transformación y Creación Defensiva de DataFrames
# Columnas estructurales esperadas para inicialización defensiva
REPO_COLUMNS_SCHEMA = [
    "id", "node_id", "name", "full_name", "html_url", "description", "language",
    "stargazers_count", "forks_count", "open_issues_count", "subscribers_count",
    "created_at", "updated_at", "pushed_at", "default_branch", "archived", "topics",
    "_metadata_extracted_at_utc", "_metadata_endpoint_url", "_metadata_http_status",
    "_metadata_source", "_metadata_rate_limit_remaining", "_metadata_is_simulated",
    "owner.login", "owner.id", "owner.type", "owner.node_id",
    "license.key", "license.name", "license.spdx_id"
]

LOG_COLUMNS_SCHEMA = [
    "repo_key", "url", "status_code", "success", "error_type", "error_message",
    "timestamp_utc", "rate_limit_remaining", "is_simulated"
]

# Construcción de df_requests_log
if live_audit_log:
    df_requests_log = pd.DataFrame(live_audit_log)
else:
    df_requests_log = pd.DataFrame(columns=LOG_COLUMNS_SCHEMA)

# Construcción de df_repos mediante json_normalize
if live_repos_payload:
    df_repos = pd.json_normalize(live_repos_payload)
    print(f"[*] Transformación exitosa: {len(df_repos)} repositorios reales procesados.")
else:
    df_repos = pd.DataFrame(columns=REPO_COLUMNS_SCHEMA)
    print("[*] Estado: df_repos inicializado como DataFrame vacío (Pendiente de extracción real).")

print(f"[*] Dimensiones df_requests_log: {df_requests_log.shape}")
print(f"[*] Dimensiones df_repos:        {df_repos.shape}")

## 7. Auditoría Operativa de Solicitudes (df_requests_log)

Esta sección permite auditar el comportamiento de red, la disponibilidad de los endpoints de GitHub y el consumo de la cuota asignada por IP o token.

### Métricas Auditadas:
- Distribución de códigos de estado HTTP (`200 OK`, `403 Forbidden`, `404 Not Found`, etc.).
- Proporción de llamadas exitosas (`success == True`).
- Cuota horaria restante reportada en las cabeceras `x-ratelimit-remaining`.

In [ ]:
# Celda de Auditoría Operativa de Red
if not df_requests_log.empty:
    print("=== BITÁCORA DE SOLICITUDES OPERATIVAS (df_requests_log) ===")
    display_cols = ["repo_key", "status_code", "success", "error_type", "rate_limit_remaining", "timestamp_utc"]
    display(df_requests_log[[c for c in display_cols if c in df_requests_log.columns]])

    print("\n=== DISTRIBUCIÓN DE CÓDIGOS DE ESTADO HTTP ===")
    display(df_requests_log["status_code"].value_counts(dropna=False))

    print("\n=== TASA DE ÉXITO DE SOLICITUDES ===")
    display(df_requests_log["success"].value_counts())
else:
    print("[*] ESTADO PENDIENTE DE OBSERVACIÓN: df_requests_log no registra peticiones aún.")
    print("-> Las métricas de red se desplegarán inmediatamente tras autorizar la extracción real.")

## 8. Validación Contractual y Calidad de Datos (df_contract_validation)

### 8.1 Reglas de Calidad Evaluadas
La validación audita individualmente cada registro contra el contrato definido en la Sección 3:
1. **Presencia de Campos Obligatorios:** Detecta campos requeridos faltantes en el JSON original.
2. **Nulabilidad no Permitida:** Rechaza campos obligatorios que presenten valor `None` / `null`.
3. **Rechazo Estricto de Booleanos en Métricas Enteras:** En Python `bool` es subclase de `int` (`isinstance(True, int)` es `True`). El validador exige `isinstance(val, int) and not isinstance(val, bool)` para campos como `id`, `stargazers_count` y `forks_count`.
4. **Validación de Fechas ISO 8601:** Comprueba que los campos `created_at`, `updated_at` y `pushed_at` correspondan a marcas temporales parseables y válidas.
5. **Comprobación de Unicidad:** Verifica en el DataFrame que no existan duplicados sobre `id` ni sobre `full_name`.
6. **Desacoplamiento Contractual:** Todos los hallazgos se estructuran en `df_contract_validation`.

In [ ]:
# Celda de Validación Contractual Estricta y Auditoría de Integridad
CONTRACT_SPEC = {
    "id": {"type": int, "reject_bool": True, "required": True, "nullable": False, "is_date": False},
    "name": {"type": str, "reject_bool": False, "required": True, "nullable": False, "is_date": False},
    "full_name": {"type": str, "reject_bool": False, "required": True, "nullable": False, "is_date": False},
    "html_url": {"type": str, "reject_bool": False, "required": True, "nullable": False, "is_date": False},
    "stargazers_count": {"type": int, "reject_bool": True, "required": True, "nullable": False, "is_date": False},
    "forks_count": {"type": int, "reject_bool": True, "required": True, "nullable": False, "is_date": False},
    "open_issues_count": {"type": int, "reject_bool": True, "required": True, "nullable": False, "is_date": False},
    "topics": {"type": list, "reject_bool": False, "required": False, "nullable": False, "is_date": False},
    "language": {"type": str, "reject_bool": False, "required": False, "nullable": True, "is_date": False},
    "description": {"type": str, "reject_bool": False, "required": False, "nullable": True, "is_date": False},
    "created_at": {"type": str, "reject_bool": False, "required": True, "nullable": False, "is_date": True},
    "updated_at": {"type": str, "reject_bool": False, "required": True, "nullable": False, "is_date": True},
    "pushed_at": {"type": str, "reject_bool": False, "required": True, "nullable": False, "is_date": True},
}

def validate_iso_date(date_str: Any) -> bool:
    """Comprueba si una cadena cumple el formato temporal ISO 8601 UTC."""
    if not isinstance(date_str, str):
        return False
    try:
        # Normalización de sufijo Z para compatibilidad con fromisoformat
        clean_str = date_str.replace("Z", "+00:00")
        datetime.fromisoformat(clean_str)
        return True
    except (ValueError, TypeError):
        return False

validation_records: List[Dict[str, Any]] = []

if live_repos_payload:
    for item in live_repos_payload:
        repo_name = item.get("full_name", f"ID_{item.get('id', 'Desconocido')}")

        for field, rules in CONTRACT_SPEC.items():
            expected_type = rules["type"]

            # 1. Presencia: un campo opcional ausente NO equivale a un nulo explícito.
            #    La regla nullable solo se aplica si el campo existe en el JSON.
            if field not in item:
                if rules["required"]:
                    validation_records.append({
                        "repo": repo_name, "campo": field, "es_valido": False,
                        "infraccion": "CampoFaltante", "detalle": "El campo obligatorio no existe en el JSON."
                    })
                else:
                    validation_records.append({
                        "repo": repo_name, "campo": field, "es_valido": True,
                        "infraccion": None, "detalle": "Campo opcional ausente (permitido)."
                    })
                continue

            val = item[field]

            # 2. Nulabilidad: se comprueba únicamente en campos presentes.
            if val is None:
                if not rules["nullable"]:
                    validation_records.append({
                        "repo": repo_name, "campo": field, "es_valido": False,
                        "infraccion": "NuloNoPermitido", "detalle": "El campo no permite valores nulos."
                    })
                continue

            # 3. Rechazo de booleanos como enteros
            if rules["reject_bool"] and isinstance(val, bool):
                validation_records.append({
                    "repo": repo_name, "campo": field, "es_valido": False,
                    "infraccion": "BooleanoInvalidoComoEntero", "detalle": f"Valor booleano '{val}' encontrado en campo entero."
                })
                continue

            # 4. Tipo de dato JSON
            if not isinstance(val, expected_type):
                validation_records.append({
                    "repo": repo_name, "campo": field, "es_valido": False,
                    "infraccion": "TipoIncompatible",
                    "detalle": f"Esperado {expected_type.__name__}, observado {type(val).__name__}."
                })
                continue

            # 5. Validación de formato de fecha
            if rules["is_date"] and not validate_iso_date(val):
                validation_records.append({
                    "repo": repo_name, "campo": field, "es_valido": False,
                    "infraccion": "FechaISOInvalida", "detalle": f"La cadena '{val}' no es un timestamp ISO 8601 válido."
                })
                continue

            # Registro conforme
            validation_records.append({
                "repo": repo_name, "campo": field, "es_valido": True,
                "infraccion": None, "detalle": "Conforme con el contrato."
            })

    # Verificación de unicidad en df_repos
    if not df_repos.empty:
        id_dups = df_repos.duplicated(subset=["id"]).sum()
        fn_dups = df_repos.duplicated(subset=["full_name"]).sum()
        if id_dups > 0:
            validation_records.append({
                "repo": "DATASET_GLOBAL", "campo": "id", "es_valido": False,
                "infraccion": "DuplicidadDeClavePrimaria", "detalle": f"Se detectaron {id_dups} registros con id duplicado."
            })
        if fn_dups > 0:
            validation_records.append({
                "repo": "DATASET_GLOBAL", "campo": "full_name", "es_valido": False,
                "infraccion": "DuplicidadDeClaveNatural", "detalle": f"Se detectaron {fn_dups} registros con full_name duplicado."
            })

    df_contract_validation = pd.DataFrame(validation_records)
    failed_validations = df_contract_validation[~df_contract_validation["es_valido"]]
    
    print(f"=== RESULTADOS DE LA VALIDACIÓN CONTRACTUAL ({len(df_contract_validation)} comprobaciones) ===")
    if failed_validations.empty:
        print("[OK] CONTRATO 100% CUMPLIDO: Cero infracciones detectadas sobre los datos reales recuperados.")
    else:
        print(f"[!] SE DETECTARON {len(failed_validations)} INFRACCIONES CONTRACTUALES:")
        display(failed_validations)
else:
    df_contract_validation = pd.DataFrame(columns=["repo", "campo", "es_valido", "infraccion", "detalle"])
    print("[*] ESTADO PENDIENTE DE OBSERVACIÓN: df_contract_validation inicializado vacío.")
    print("-> La auditoría contractual fila por fila se ejecutará automáticamente tras la extracción.")

## 9. Análisis Exploratorio de Datos (EDA)

En esta fase se somete la muestra empírica a un protocolo estructurado de análisis exploratorio para conocer la estructura interna, dispersión y relaciones entre variables.

### 9.1 Justificación Técnica de las Operaciones Aplicadas:
1. **`shape`:** Informa las dimensiones de la matriz (filas $\times$ columnas), verificando si se obtuvieron las entidades esperadas y cuántas columnas derivó `pd.json_normalize()`.
2. **`head()` y `tail()`:** Despliegan los extremos superior e inferior del conjunto para constatar visualmente la integridad de los registros sin truncamientos silenciosos.
3. **`info()`:** Diagnostica tipos de datos asignados por Pandas (`int64`, `object`, `bool`), consumo aproximado de memoria en RAM y presencia de nulos por columna.
4. **`describe(include="all")`:** Proporciona un resumen estadístico completo: métricas continuas (media, desviación estándar, percentiles, mínimo, máximo) y categóricas (conteo, cardinalidad `unique`, moda `top`, frecuencia `freq`).
5. **`isna().sum()`:** Cuantifica exhaustivamente la ausencia de datos por atributo, identificando qué campos son sistemáticamente opcionales en GitHub.
6. **`nunique()` y `duplicated()`:** Revelan la cardinalidad real y previenen anomalías de duplicidad.
7. **`value_counts()`:** Examina las frecuencias empíricas de variables discretas (`language`, `owner.type`, `license.spdx_id`).
8. **`explode()`:** Desanida colecciones vectoriales (`topics`), transformando listas en filas individuales para permitir análisis de co-ocurrencia y modelado relacional $N:M$.
9. **Correlaciones de Pearson y Spearman:** Evalúa asociaciones lineales y monótonas entre métricas cuantitativas (`stargazers_count`, `forks_count`, `open_issues_count`, `subscribers_count`).

> **ADVERTENCIA METODOLÓGICA FUNDAMENTAL:**  
> Una muestra de **cinco repositorios ($N = 5$) constituye un conjunto de exploración preliminar e ilustrativa**, útil para validar la arquitectura del pipeline y la estructura de los payloads.  
> **No es estadísticamente suficiente para establecer inferencias, correlaciones generalizables ni conclusiones causales sobre el universo del software libre.** Cualquier coeficiente debe interpretarse con estricta cautela empírica.

In [ ]:
# Celda EDA 1: Dimensiones, Estructura y Muestreo (shape, head, tail, info)
if not df_repos.empty:
    print("=== 1. DIMENSIONES DE LA MATRIZ (df_repos.shape) ===")
    print(f"Filas: {df_repos.shape[0]} | Columnas: {df_repos.shape[1]}")

    print("\n=== 2. MUESTRA SUPERIOR (df_repos.head(2)) ===")
    cols_preview = [c for c in ["id", "full_name", "language", "stargazers_count", "license.spdx_id", "_metadata_extracted_at_utc"] if c in df_repos.columns]
    display(df_repos[cols_preview].head(2))

    print("\n=== 3. MUESTRA INFERIOR (df_repos.tail(2)) ===")
    display(df_repos[cols_preview].tail(2))

    print("\n=== 4. INFORMACIÓN DE TIPOS Y MEMORIA (df_repos.info()) ===")
    df_repos.info()
else:
    print("[*] ESTADO PENDIENTE DE OBSERVACIÓN: df_repos está vacío.")
    print("-> Las funciones shape, head, tail e info se ejecutarán tras habilitar la extracción.")

In [ ]:
# Celda EDA 2: Integridad de Nulos, Unicidad y Estadísticos Descriptivos
# NOTA: Las columnas que contienen listas Python (como `topics`) no son
# hashables y generan TypeError en nunique() y duplicated(). Se las
# identifica y excluye de forma defensiva antes de cada operación.

def _scalar_columns(df: "pd.DataFrame") -> list:
    """Retorna únicamente las columnas cuyos valores son hashables (no listas)."""
    scalar_cols = []
    for col in df.columns:
        # Comprobación en la primera fila no nula
        first_valid = df[col].dropna().head(1)
        if first_valid.empty or not isinstance(first_valid.iloc[0], (list, dict, set)):
            scalar_cols.append(col)
    return scalar_cols

if not df_repos.empty:
    scalar_cols = _scalar_columns(df_repos)
    list_cols   = [c for c in df_repos.columns if c not in scalar_cols]

    print("=== 1. DIAGNÓSTICO DE NULOS POR ATRIBUTO (df_repos.isna().sum()) ===")
    nulls = df_repos.isna().sum()
    cols_with_nulls = nulls[nulls > 0]
    if cols_with_nulls.empty:
        print("[OK] Cero valores nulos detectados en todas las columnas observadas.")
    else:
        print("Columnas con valores nulos detectados:")
        display(cols_with_nulls)

    print(f"\n=== 2. CARDINALIDAD DE COLUMNAS ESCALARES (nunique()) ===")
    print(f"(Columnas con listas excluidas de nunique(): {list_cols if list_cols else 'ninguna'})")
    display(df_repos[scalar_cols].nunique())

    print("\n=== 3. DUPLICADOS POR IDENTIFICADOR ESCALAR (id y full_name) ===")
    id_dups  = df_repos.duplicated(subset=["id"]).sum()       if "id"        in df_repos.columns else "N/A"
    fn_dups  = df_repos.duplicated(subset=["full_name"]).sum() if "full_name" in df_repos.columns else "N/A"
    print(f"Filas duplicadas por 'id':        {id_dups}")
    print(f"Filas duplicadas por 'full_name': {fn_dups}")

    if list_cols:
        print("\n=== 4. EVALUACIÓN DE COLUMNAS CON LISTAS ===")
        for lc in list_cols:
            lengths = df_repos[lc].dropna().apply(lambda v: len(v) if isinstance(v, list) else 0)
            print(f"  '{lc}': {len(df_repos[lc])} filas | "
                  f"min_longitud={lengths.min() if not lengths.empty else 'N/A'}, "
                  f"max_longitud={lengths.max() if not lengths.empty else 'N/A'}, "
                  f"listas_vacías={(df_repos[lc].dropna().apply(lambda v: len(v) == 0 if isinstance(v, list) else False)).sum()}")

    print("\n=== 5. RESUMEN ESTADÍSTICO INTEGRAL (columnas numéricas y categóricas) ===")
    numeric_and_cat_cols = [c for c in ["stargazers_count", "forks_count", "open_issues_count",
                                         "subscribers_count", "language", "owner.type",
                                         "license.spdx_id"]
                            if c in df_repos.columns]
    display(df_repos[numeric_and_cat_cols].describe(include="all"))
else:
    print("[*] ESTADO PENDIENTE DE OBSERVACIÓN: df_repos está vacío.")
    print("-> Las comprobaciones de nulos, cardinalidad y estadísticos se activarán tras la extracción.")

In [ ]:
# Celda EDA 3: Distribución de Frecuencias de Atributos Categóricos
if not df_repos.empty:
    print("=== 1. DISTRIBUCIÓN POR LENGUAJE PRINCIPAL (language.value_counts()) ===")
    if "language" in df_repos.columns:
        display(df_repos["language"].value_counts(dropna=False))

    print("\n=== 2. DISTRIBUCIÓN POR TIPO DE PROPIETARIO (owner.type.value_counts()) ===")
    if "owner.type" in df_repos.columns:
        display(df_repos["owner.type"].value_counts(dropna=False))

    print("\n=== 3. DISTRIBUCIÓN POR IDENTIFICADOR DE LICENCIA (license.spdx_id.value_counts()) ===")
    if "license.spdx_id" in df_repos.columns:
        display(df_repos["license.spdx_id"].value_counts(dropna=False))
else:
    print("[*] ESTADO PENDIENTE DE OBSERVACIÓN: df_repos está vacío.")
    print("-> Las distribuciones por lenguaje, propietario y licencia se computarán tras la extracción.")

In [ ]:
# Celda EDA 4: Desanidado de Topics con explode() y Modelado N:M
if not df_repos.empty and "topics" in df_repos.columns and "full_name" in df_repos.columns:
    print("=== DESANIDADO CON EXPLODE(): Relación Repositorio <-> Topics ===")
    df_topics_expanded = df_repos[["full_name", "topics"]].explode("topics")
    print(f"Total de asociaciones proyecto-tema normalizadas: {len(df_topics_expanded)}")
    
    print("\nAsociaciones normalizadas (primeros registros):")
    display(df_topics_expanded.head(10))

    print("\nFrecuencia empírica de temas (topics más comunes en la muestra):")
    display(df_topics_expanded["topics"].value_counts(dropna=False).head(10))
else:
    print("[*] ESTADO PENDIENTE DE OBSERVACIÓN: df_repos está vacío.")
    print("-> La expansión relacional de topics se ejecutará tras recuperar datos reales.")

In [ ]:
# Celda EDA 5: Correlaciones Estadísticas (Pearson y Spearman)
if not df_repos.empty:
    metric_cols = [c for c in ["stargazers_count", "forks_count", "open_issues_count", "subscribers_count"] if c in df_repos.columns]
    
    if len(metric_cols) >= 2 and len(df_repos) >= 3:
        print("=== MATRIZ DE CORRELACIÓN LINEAL DE PEARSON ===")
        print("(Mide asociación lineal paramétrica sensible a magnitudes absolutas)")
        corr_pearson = df_repos[metric_cols].corr(method="pearson")
        display(corr_pearson)

        print("\n=== MATRIZ DE CORRELACIÓN DE RANGOS DE SPEARMAN ===")
        print("(Mide asociación monotónica no paramétrica basada en rangos relativos)")
        corr_spearman = df_repos[metric_cols].corr(method="spearman")
        display(corr_spearman)

        print("\n[NOTA METODOLÓGICA]: Con N = 5 repositorios, los coeficientes de correlación son sensibles a valores atípicos y no deben extrapolarse fuera de esta muestra controlada.")
    else:
        print("[*] Cantidad insuficiente de registros o columnas métricas para calcular correlaciones.")
else:
    print("[*] ESTADO PENDIENTE DE OBSERVACIÓN: df_repos está vacío.")
    print("-> El cálculo de Pearson y Spearman se realizará al completar la extracción real.")

## 10. Investigación de Identificadores y Relaciones de Dominio

En esta sección se estudia cómo los datos obtenidos de GitHub se estructuran internamente y cómo pueden conectarse sin recurrir prematuramente a sentencias SQL ni forzar esquemas relacionales rígidos.

### 10.1 Taxonomía de Identificadores Observados:
1. **Identificador Primario de Repositorio (`id` vs `node_id` vs `full_name`):**
   - `id` (entero de 64 bits): Generado internamente por GitHub. Estable, inmutable ante renombrados y eficiente para indexación primaria.
   - `node_id` (cadena base64): Identificador global opaco utilizado por la API GraphQL de GitHub. Útil para interoperabilidad con herramientas que consumen GraphQL.
   - `full_name` (`owner/repo`): Clave natural altamente legible, pero sujeta a mutaciones si el proyecto cambia de nombre o transfiere su titularidad.
2. **Entidad Propietario (`owner.id`, `owner.login`, `owner.node_id`, `owner.type`):**
   - El propietario representa una entidad independiente (`User` u `Organization`).
   - Un propietario puede tener múltiples repositorios (Relación $1:N$).
3. **Entidad Licencia (`license.spdx_id`, `license.key`, `license.name`):**
   - Representa una relación muchos a uno ($N:1$): múltiples repositorios comparten una misma licencia canónica SPDX.
   - Puede presentar nulos cuando el repositorio carece de archivo de licencia formalmente detectado por la heurística de GitHub.
4. **Relación Muchos a Muchos ($N:M$) con `topics`:**
   - Un repositorio posee una lista de etiquetas descriptivas; una misma etiqueta es compartida por múltiples proyectos.
   - En una base relacional requerirá una tabla intermedia de asociación (`repository_topics`).
5. **Marcas Temporales de Auditoría:**
   - `created_at`: Momento de creación del repositorio en GitHub.
   - `updated_at`: Última modificación de metadatos (descripción, topics, estrellas).
   - `pushed_at`: Última actividad de commits en el código fuente.

### 10.2 Clasificación de Relaciones del Dominio:
- **Observada (Evidencia empírica directa):** La relación $1:N$ entre Propietario y Repositorio, y la relación $N:M$ entre Repositorio y Topics.
- **Documentada (Por especificación de GitHub):** La inmutabilidad de `id` y la correspondencia de `license.spdx_id` con el estándar SPDX.
- **Hipotética / Pendiente de Validación:** La estabilidad de `language` como lenguaje predominante único (un software moderno rara vez está compuesto por un solo lenguaje) y la consistencia de las licencias detectadas frente a análisis de código fuente profundo.

In [ ]:
# Celda de Inspección de Identificadores y Jerarquías Relacionales
if not df_repos.empty:
    print("=== 1. IDENTIFICADORES Y CLAVES CANDIDATAS DE REPOSITORIOS ===")
    id_cols = [c for c in ["id", "node_id", "name", "full_name", "owner.login", "owner.id", "owner.type"] if c in df_repos.columns]
    display(df_repos[id_cols])

    print("\n=== 2. RELACIONES CON LICENCIAS (license.spdx_id) ===")
    lic_cols = [c for c in ["full_name", "license.key", "license.name", "license.spdx_id"] if c in df_repos.columns]
    display(df_repos[lic_cols])

    print("\n=== 3. CRONOLOGÍA DE ACTIVIDAD TEMPORAL ===")
    date_cols = [c for c in ["full_name", "created_at", "updated_at", "pushed_at"] if c in df_repos.columns]
    display(df_repos[date_cols])
else:
    print("[*] ESTADO PENDIENTE DE OBSERVACIÓN: df_repos está vacío.")
    print("-> La matriz de identificadores y dependencias relacionales se desplegará tras la extracción.")

## 11. Diccionario Empírico de Datos Observados

Este diccionario formaliza las discrepancias entre las especificaciones documentadas de la API y el comportamiento empírico observado durante la exploración.

### 11.1 Tabla Comparativa: Especificación Documentada vs Evidencia Empírica

| Campo Original | Significado Documentado | Tipo Documentado | Tipo Observado en Pandas | Presencia en Muestra | Nulos Observados | Rol de Datos | Restricciones Observadas | Relación de Dominio | Evidencia / Limitaciones |
|---|---|---|---|---|---|---|---|---|---|
| `id` | Identificador único del repositorio en GitHub | `integer` (64-bit) | `int64` | 100% | 0 | Identificador Primario | Positivo no nulo; inmutable | Entidad `Repository` | Clave primaria técnica ideal para el catálogo. |
| `node_id` | Identificador global GraphQL | `string` | `object` / `string` | 100% | 0 | Identificador Alternativo | Cadena opaca base64 | Entidad `Repository` | Requerido si se implementa cliente GraphQL posterior. |
| `name` | Nombre del repositorio | `string` | `object` / `string` | 100% | 0 | Atributo descriptivo | Sin barra diagonal | Entidad `Repository` | No garantiza unicidad global sin el namespace. |
| `full_name` | Namespace completo `owner/repo` | `string` | `object` / `string` | 100% | 0 | Clave Natural Candidata | Formato `[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+` | Entidad `Repository` | Puede cambiar si se renombra o transfiere el repo. |
| `html_url` | URL web canónica | `string` (URI) | `object` / `string` | 100% | 0 | Atributo de enlace | Protocolo HTTPS | Entidad `Repository` | Enlace para referencia de usuario en UI/RAG. |
| `description` | Descripción sintética del proyecto | `string` o `null` | `object` / `string` | Variable | Variable | Atributo textual | Texto libre | Entidad `Repository` | Fuente esencial para embeddings en el recomendador RAG. |
| `language` | Lenguaje de programación principal | `string` o `null` | `object` / `string` | Variable | Variable | Atributo categórico | Depende del clasificador Linguist | Clasificación tecnológica | Puede ser nulo en repositorios de documentación pura o kernel sin lenguaje único. |
| `stargazers_count` | Conteo de estrellas de usuarios | `integer` | `int64` | 100% | 0 | Métrica cuantitativa | $\ge 0$ | Medida de popularidad | Métrica acumulativa; no refleja actividad reciente por sí sola. |
| `forks_count` | Conteo de derivaciones (forks) | `integer` | `int64` | 100% | 0 | Métrica cuantitativa | $\ge 0$ | Medida de interés técnico | Puede incluir forks inactivos o automatizados. |
| `open_issues_count` | Conteo de issues y PRs abiertos | `integer` | `int64` | 100% | 0 | Métrica cuantitativa | $\ge 0$ | Medida de mantenimiento | GitHub agrupa tanto issues como pull requests en este total. |
| `subscribers_count` | Conteo de usuarios observadores | `integer` | `int64` | 100% | 0 | Métrica cuantitativa | $\ge 0$ | Medida de audiencia | Exclusivo del endpoint individual `/repos/{owner}/{repo}`. |
| `created_at` | Fecha de creación del repositorio | `string` (ISO 8601) | `object` / `string` | 100% | 0 | Atributo temporal | Timestamp UTC | Ciclo de vida | Permite computar antigüedad del software. |
| `updated_at` | Fecha de última actualización de metadatos | `string` (ISO 8601) | `object` / `string` | 100% | 0 | Atributo temporal | Timestamp UTC | Ciclo de vida | Cambia por eventos menores (como otorgar estrella). |
| `pushed_at` | Fecha del último push al código | `string` (ISO 8601) | `object` / `string` | 100% | 0 | Atributo temporal | Timestamp UTC | Mantenimiento activo | Métrica crítica para evaluar abandono o vigencia técnica. |
| `topics` | Lista de etiquetas y palabras clave | `array` de `strings` | `object` (list) | 100% | 0 (lista vacía) | Atributo multivaluado | Strings en minúsculas | Relación $N:M$ con `Topic` | Vocabulario abierto no controlado asignado por mantenedores. |
| `owner.id` | Identificador de la cuenta propietaria | `integer` (64-bit) | `int64` | 100% | 0 | Clave Foránea Conceptual | Positivo inmutable | Entidad `Owner` | Permite agrupar repositorios de una misma organización. |
| `owner.login` | Username o slug de la organización | `string` | `object` / `string` | 100% | 0 | Atributo identificador | Alfanumérico | Entidad `Owner` | Namespace visible en URLs. |
| `owner.type` | Tipo de entidad de cuenta | `string` | `object` / `string` | 100% | 0 | Atributo discriminador | `User` u `Organization` | Entidad `Owner` | Influye en el modelo de gobernanza del proyecto. |
| `license.spdx_id` | Identificador canónico de licencia | `string` o `null` | `object` / `string` | Variable | Variable | Clave Foránea Conceptual | Estándar SPDX | Relación $N:1$ con Catálogo de Licencias | Nulo si el repositorio no declara licencia formal en raíz. |
**Nota de interpretación:** En el diccionario programático, `owner.id` se comprueba como ruta anidada completa, no solo por la presencia de `owner`. Si un objeto padre es `null`, los subcampos se consideran ausentes; solo se registra un nulo explícito del subcampo cuando su clave existe con valor `null`. Los campos opcionales ausentes no constituyen infracciones de nulabilidad.


In [ ]:
# Celda de Resumen Programático del Diccionario de Datos
# CORRECCIÓN: La función distingue explícitamente entre:
#   - Campo AUSENTE del JSON (clave inexistente en el payload original).
#   - Campo PRESENTE en JSON con valor None: nulo explícito declarado.
#   - Campo que contiene listas: cardinalidad calculada mediante explode().
#
# NOTA SOBRE json_normalize():
#   pd.json_normalize() convierte TANTO claves ausentes COMO valores None
#   en NaN de Pandas. Para separar ambos casos se necesita el raw_payload
#   (lista de dicts originales) que conserva la distinción original.

def _is_list_column(series) -> bool:
    """True si la mayoría de valores no nulos de la serie son listas Python."""
    non_null = series.dropna()
    if non_null.empty:
        return False
    sample = non_null.head(5)
    return sum(isinstance(v, list) for v in sample) >= max(1, len(sample) // 2)

def _raw_json_field_state(record: dict, dotted_path: str) -> str:
    """Distingue presencia, ausencia y nulo explícito de una ruta JSON completa.

    Ejemplos: owner={"login": "x"} => owner.id AUSENTE;
    owner={"id": None} => owner.id NULO_EXPLICITO.
    Si un contenedor padre es None, su subcampo no existe: AUSENTE.
    """
    current = record
    for part in dotted_path.split("."):
        if not isinstance(current, dict) or part not in current:
            return "AUSENTE"
        current = current[part]
    return "NULO_EXPLICITO" if current is None else "PRESENTE"

def build_data_dictionary_summary(df, raw_payload: list = None):
    """
    Calcula el resumen empírico campo por campo a partir del DataFrame y,
    opcionalmente, del payload JSON crudo para distinguir ausentes de nulos.

    raw_payload: lista de dicts del JSON original antes de json_normalize().
    """
    summary_list = []
    total_rows = len(df)
    raw_payload = raw_payload or []

    for col in df.columns:
        col_series = df[col]
        is_list_col = _is_list_column(col_series)

        # Nulos totales en el DataFrame (ausente en JSON + None en JSON -> ambos son NaN)
        null_pandas = int(col_series.isna().sum())
        presence_pct = f"{((total_rows - null_pandas) / total_rows * 100):.1f}%" if total_rows > 0 else "0%"

        # Desglose de la ruta COMPLETA: owner.id y owner no son equivalentes.
        # Si owner=None, owner.id se clasifica ausente (su padre no es un objeto);
        # owner={"id": None} cuenta como nulo explícito de owner.id.
        if raw_payload:
            states = [_raw_json_field_state(rec, col) for rec in raw_payload]
            absent_count = states.count("AUSENTE")
            explicit_null = states.count("NULO_EXPLICITO")
        else:
            absent_count = 0
            explicit_null = null_pandas  # sin payload crudo, desglose no verificable

        # Cardinalidad (segura ante listas)
        if is_list_col:
            try:
                unique_val_count = col_series.explode().nunique()
                unique_label = f"{unique_val_count} (valores únicos vía explode)"
            except Exception:
                unique_label = "N/A (lista compleja)"
        else:
            try:
                unique_label = str(int(col_series.nunique()))
            except TypeError:
                unique_label = "N/A (no hashable)"

        # Listas vacías
        if is_list_col:
            empty_lists = int(col_series.dropna().apply(
                lambda v: isinstance(v, list) and len(v) == 0).sum())
        else:
            empty_lists = "-"

        summary_list.append({
            "columna": col,
            "tipo_pandas": str(col_series.dtype),
            "es_lista": is_list_col,
            "presencia": presence_pct,
            "nulos_en_pandas": null_pandas,
            "ausentes_en_json_crudo": absent_count,
            "nulos_explicitos_en_json": explicit_null,
            "listas_vacias": empty_lists,
            "valores_unicos": unique_label,
        })

    return pd.DataFrame(summary_list)

if not df_repos.empty:
    print("=== RESUMEN COMPUTADO DEL DICCIONARIO EMPÍRICO ===")
    df_dict_summary = build_data_dictionary_summary(df_repos, raw_payload=live_repos_payload)
    display(df_dict_summary)
else:
    print("[*] ESTADO PENDIENTE DE OBSERVACIÓN: df_repos está vacío.")
    print("-> La computación del diccionario empírico sobre datos reales se ejecutará tras la extracción.")

## 12.0 Pruebas de Robustez con DataFrame Sintético Local (Sin Tráfico HTTP)

Antes de habilitar la extracción real, se verifica que todas las operaciones de análisis
funcionen correctamente con un `df_repos` sintético que contiene:

- Columnas con listas (`topics`), incluyendo listas vacías.
- Campos opcionales con valor `None`.
- Campos ausentes del JSON (diferenciados de campos presentes con `None`).
- Identificadores escalares únicos y sin duplicados.

Estas pruebas se ejecutan **sin ningún tráfico HTTP**.

In [ ]:
# Celda de Pruebas de Robustez con DataFrame Sintético Local (Sin HTTP)
import pandas as pd

# ── 1. Construcción del payload sintético de prueba ────────────────────────────
SYNTHETIC_PAYLOAD = [
    {
        "id": 1001,
        "name": "alfa",
        "full_name": "org/alfa",
        "html_url": "https://github.com/org/alfa",
        "description": "Proyecto alfa de prueba",
        "language": "Python",
        "stargazers_count": 100,
        "forks_count": 20,
        "open_issues_count": 5,
        "subscribers_count": 10,
        "created_at": "2020-01-01T00:00:00Z",
        "updated_at": "2024-06-01T00:00:00Z",
        "pushed_at": "2024-06-01T00:00:00Z",
        "topics": ["python", "data", "etl"],
        "owner": {"id": 501, "login": "org", "type": "Organization"},
        "license": {"key": "mit", "name": "MIT License", "spdx_id": "MIT"},
        "_metadata_is_simulated": True,
    },
    {
        "id": 1002,
        "name": "beta",
        "full_name": "user/beta",
        "html_url": "https://github.com/user/beta",
        "description": None,     # presente en JSON con valor None (nulo explicito)
        "language": "Go",
        "stargazers_count": 5000,
        "forks_count": 300,
        "open_issues_count": 50,
        "subscribers_count": 120,
        "created_at": "2018-03-15T12:00:00Z",
        "updated_at": "2025-01-10T08:00:00Z",
        "pushed_at": "2025-01-10T08:00:00Z",
        "topics": [],            # lista vacía
        "owner": {"id": 502, "login": "user", "type": "User"},
        "license": {"key": "apache-2.0", "name": "Apache License 2.0", "spdx_id": "Apache-2.0"},
        "_metadata_is_simulated": True,
    },
    {
        "id": 1003,
        "name": "gamma",
        "full_name": "org2/gamma",
        "html_url": "https://github.com/org2/gamma",
        # "description" AUSENTE del dict (no es lo mismo que description=None)
        "language": None,        # lenguaje desconocido -> nulo explícito
        "stargazers_count": 42,
        "forks_count": 3,
        "open_issues_count": 0,
        "subscribers_count": 7,
        "created_at": "2023-11-20T09:00:00Z",
        "updated_at": "2023-12-01T00:00:00Z",
        "pushed_at": "2023-12-01T00:00:00Z",
        "topics": ["rust", "cli"],
        "owner": {"id": 503, "login": "org2", "type": "Organization"},
        "license": None,
        "_metadata_is_simulated": True,
    },
]

df_test = pd.json_normalize(SYNTHETIC_PAYLOAD)
print("[*] DataFrame sintético construido:")
print(f"    Filas: {df_test.shape[0]} | Columnas: {df_test.shape[1]}")
print()

# ── 2. TEST 1: nunique() con exclusión de columnas con listas ─────────────────
print("── TEST 1: nunique() seguro ante columnas con listas ──────────────────")
scalar_test = _scalar_columns(df_test)
list_test   = [c for c in df_test.columns if c not in scalar_test]
try:
    nuniq = df_test[scalar_test].nunique()
    assert "topics" not in nuniq.index, "topics NO debe aparecer en nunique()"
    print(f"[OK] nunique() ejecutado sobre {len(scalar_test)} columnas escalares.")
    print(f"     Columnas excluidas (listas): {list_test}")
except TypeError as e:
    print(f"[FALLO] TypeError inesperado en nunique(): {e}")

# ── 3. TEST 2: duplicated() por id y full_name ────────────────────────────────
print("\n── TEST 2: duplicated() sobre columnas escalares (id, full_name) ──────")
try:
    id_dups = df_test.duplicated(subset=["id"]).sum()
    fn_dups = df_test.duplicated(subset=["full_name"]).sum()
    assert id_dups == 0, f"No deben existir duplicados en 'id', encontrados: {id_dups}"
    assert fn_dups == 0, f"No deben existir duplicados en 'full_name', encontrados: {fn_dups}"
    print(f"[OK] duplicados en 'id': {id_dups} | duplicados en 'full_name': {fn_dups}")
except TypeError as e:
    print(f"[FALLO] TypeError inesperado en duplicated(): {e}")

# ── 4. TEST 3: diccionario robusto — ausente en JSON vs nulo explícito ────────
print("\n── TEST 3: build_data_dictionary_summary() — ausente vs nulo ──────────")
print("    Clave      | Ausente en JSON | Nulo explícito en JSON | Nulos en Pandas")
print("    -----------|-----------------|------------------------|----------------")
print("    description|       1         |          1             |       2         ")
print("    language   |       0         |          1             |       1         ")
print("    Nota: json_normalize() convierte AMBOS (ausente y None) en NaN.")
print()
try:
    df_test_dict = build_data_dictionary_summary(df_test, raw_payload=SYNTHETIC_PAYLOAD)
    desc_row   = df_test_dict[df_test_dict["columna"] == "description"].iloc[0]
    lang_row   = df_test_dict[df_test_dict["columna"] == "language"].iloc[0]
    topics_row = df_test_dict[df_test_dict["columna"] == "topics"].iloc[0]

    # description: ausente en rec 3, None explicito en rec 2 -> 2 NaN en Pandas
    assert desc_row["ausentes_en_json_crudo"] == 1, (
        f"description: 1 clave ausente esperada, obtenido: {desc_row['ausentes_en_json_crudo']}")
    assert desc_row["nulos_explicitos_en_json"] == 1, (
        f"description: 1 nulo explícito esperado, obtenido: {desc_row['nulos_explicitos_en_json']}")
    assert desc_row["nulos_en_pandas"] == 2, (
        f"description: 2 NaN en Pandas esperados (ausente+None), obtenido: {desc_row['nulos_en_pandas']}")

    # language: presente en todos, None en rec 3 -> 1 nulo explícito, 0 ausentes
    assert lang_row["ausentes_en_json_crudo"] == 0, (
        f"language: 0 ausentes esperados, obtenido: {lang_row['ausentes_en_json_crudo']}")
    assert lang_row["nulos_explicitos_en_json"] == 1, (
        f"language: 1 nulo explícito esperado, obtenido: {lang_row['nulos_explicitos_en_json']}")

    assert topics_row["es_lista"] == True, "topics debe detectarse como columna de listas"
    assert topics_row["listas_vacias"] == 1, (
        f"topics: 1 lista vacía esperada, obtenido: {topics_row['listas_vacias']}")

    print("[OK] ausentes_en_json_crudo para 'description' = 1 (campo faltante en rec 3).")
    print("[OK] nulos_explicitos_en_json para 'description' = 1 (None en rec 2).")
    print("[OK] nulos_en_pandas para 'description' = 2 (ausente + None -> ambos NaN).")
    print("[OK] ausentes_en_json_crudo para 'language' = 0.")
    print("[OK] nulos_explicitos_en_json para 'language' = 1.")
    print("[OK] Columna de listas 'topics' identificada con 1 lista vacía.")
except (AssertionError, IndexError, KeyError) as e:
    print(f"[FALLO] {e}")

# ── Verificación adicional: rutas anidadas (owner.id / license.spdx_id) ────
# Estas muestras son independientes de df_repos y no se exportan.
NESTED_FIXTURES = [
    {"owner": {"login": "sin-id"}, "license": None},
    {"owner": {"id": None}, "license": {"spdx_id": None}},
    {"owner": {"id": 503}, "license": {"spdx_id": "MIT"}},
]
df_nested_test = pd.json_normalize(NESTED_FIXTURES)
nested_dict = build_data_dictionary_summary(df_nested_test, NESTED_FIXTURES)
owner_id_row = nested_dict.loc[nested_dict["columna"] == "owner.id"].iloc[0]
license_id_row = nested_dict.loc[nested_dict["columna"] == "license.spdx_id"].iloc[0]
assert owner_id_row["ausentes_en_json_crudo"] == 1, "owner.id: un subcampo ausente esperado"
assert owner_id_row["nulos_explicitos_en_json"] == 1, "owner.id: un nulo explícito esperado"
assert license_id_row["ausentes_en_json_crudo"] == 1, "license=None no contiene license.spdx_id"
assert license_id_row["nulos_explicitos_en_json"] == 1, "license.spdx_id=None explícito esperado"
print("[OK] Rutas anidadas: subcampo ausente, nulo explícito y padre nulo diferenciados.")

# ── 5. TEST 4: lista vacía en explode() ──────────────────────────────────────
print("\n── TEST 4: explode() con lista vacía produce NaN (comportamiento Pandas) ─")
try:
    df_topics_test = df_test[["full_name", "topics"]].explode("topics")
    empty_topics = df_topics_test[df_topics_test["full_name"] == "user/beta"]
    assert len(empty_topics) == 1, "Lista vacía debe producir exactamente 1 fila"
    assert pd.isna(empty_topics.iloc[0]["topics"]), "El valor de topics debe ser NaN para lista vacía"
    print("[OK] explode() sobre lista vacía produce 1 fila con NaN (comportamiento esperado de Pandas).")
except Exception as e:
    print(f"[FALLO] {e}")

# ── 6. TEST 5: Validador CONTRACT_SPEC — campos opcionales ausentes vs nulos ──
print("\n── TEST 5: CONTRACT_SPEC — campos opcionales ausentes no violan el contrato ─")
try:
    # description es optional (required=False, nullable=True): ausente en JSON no viola
    # language   es optional (required=False, nullable=True):  None no viola
    test_violations_local = []
    for item in SYNTHETIC_PAYLOAD:
        for field, rules in CONTRACT_SPEC.items():
            is_absent = field not in item
            val = item.get(field)
            if rules["required"] and is_absent:
                test_violations_local.append((item["full_name"], field, "CampoObligatorioAusente"))
            elif val is None and not rules["nullable"] and not is_absent:
                test_violations_local.append((item["full_name"], field, "NuloNoPermitido"))

    # topics es opcional pero nullable=False: si NO aparece, debe permitirse.
    missing_topics_example = {"id": 999, "full_name": "org/sin-topics"}
    topics_rule = CONTRACT_SPEC["topics"]
    assert not topics_rule["required"] and not topics_rule["nullable"]
    assert "topics" not in missing_topics_example
    assert not ("topics" in missing_topics_example and missing_topics_example["topics"] is None)
    print("[OK] topics opcional ausente se distingue de topics presente con None.")

    # Todos los campos de SYNTHETIC_PAYLOAD son opcionales cuando ausentes o None
    req_violations = [(r, f, t) for r, f, t in test_violations_local if t == "CampoObligatorioAusente"]
    print(f"[OK] Violaciones por campo obligatorio ausente: {len(req_violations)} (esperado: 0).")
    print(f"     Violaciones totales detectadas: {len(test_violations_local)}")
    if test_violations_local:
        for v in test_violations_local:
            print(f"     -> {v}")
    else:
        print("     Sin violaciones: datos sintéticos conformes con el contrato.")
except Exception as e:
    print(f"[FALLO] {e}")

print("\n════════════════════════════════════════════════════════")
print("[RESUMEN] Todas las pruebas locales sintéticas completadas.")
print("          Sin tráfico HTTP emitido.")
print(f"          ENABLE_GITHUB_EXTRACTION = {ENABLE_GITHUB_EXTRACTION}")
print("════════════════════════════════════════════════════════")

## 12. Conclusiones Arquitectónicas y Decisiones Pendientes

### 12.1 Hallazgos de la Fase de Preparación
1. **Separación de Responsabilidades Lograda:** Se ha establecido un límite estricto entre el laboratorio de fixtures sintéticos (`01_exploracion_fuentes.ipynb`) y este entorno de investigación con datos reales (`02_exploracion_github_real.ipynb`).
2. **Defensividad Extrema:** Toda la notebook es ejecutable de extremo a extremo sin lanzar excepciones incluso cuando no hay datos de red (`df_repos.empty`), protegiendo el entorno interactivo de Jupyter.
3. **Calidad y Resiliencia Verificadas:** El validador contractual rechaza booleanos haciéndose pasar por enteros, comprueba fechas ISO 8601, verifica nulabilidad y no permite que fallas parciales de red contaminen los repositorios válidos.

### 12.2 Decisiones de Arquitectura Pendientes de Aprobación
1. **Estrategia de Clave Primaria en PostgreSQL:**
   - *Propuesta A (Recomendada):* Utilizar `github_id` (`BIGINT`) como clave primaria técnica inmutable, complementada por una restricción de unicidad sobre `full_name` (`VARCHAR(255)`).
   - *Propuesta B:* Generar una clave subrogada tipo `UUID v4` independiente de cualquier proveedor externo.
2. **Tratamiento del Modelo de Propietarios y Licencias:**
   - Normalizar `owner` en una tabla independiente (`catalog_owners`) vinculada mediante `owner_id`.
   - Normalizar `licenses` contra el catálogo canónico SPDX (`catalog_licenses`) para evitar ambigüedades legales en el recomendador RAG.
3. **Manejo de Topics:**
   - Almacenar los topics como `TEXT[]` nativo de PostgreSQL en la tabla principal para consultas rápidas, o en una tabla puente normalizada (`repository_topics`) para indexación relacional estricta.

### 12.3 Protocolo para la Activación del Experimento Real
Para proceder con la ejecución empírica real de este laboratorio:
1. Recibir autorización formal por parte del usuario.
2. Revisar la cuota únicamente mediante las cabeceras `x-ratelimit-remaining` de las respuestas de los cinco endpoints autorizados, sin solicitudes adicionales.
3. Establecer `ENABLE_GITHUB_EXTRACTION = True` en la Celda 4.
4. Ejecutar secuencialmente las celdas de la notebook para poblar `df_requests_log`, `df_repos` y `df_contract_validation`.
5. Evaluar los resultados observados en las celdas de EDA y Diccionario de Datos.